# AI Privacy: DP-SGD Training

Annotated study notebook for HTB Academy module 335. It compares ordinary CIFAR-10 training with DP-SGD and measures confidence-based membership inference. Run it in the `AI` environment after installing the HTB AI library, PyTorch, Opacus, torchvision, and safetensors. It does not submit to HTB.

**Learning goal:** connect every code block to the privacy concepts and formulas from the course.

## Core notation

`g_i` is pronounced **g sub i**: the gradient from one example. `||g_i||_2` is **the L-two norm of g sub i**, its Euclidean size. `C` is the clipping limit. `epsilon` is the privacy budget; smaller is stronger privacy. `delta` is the small failure-probability allowance.

The clipping idea is `g_i_clipped = g_i * min(1, C / ||g_i||_2)`. Read aloud: **g sub i clipped equals g sub i times the minimum of one and C divided by the L-two norm of g sub i**. If a gradient has norm 5.2 and `C = 1.0`, it is scaled down to norm 1.0 without changing direction.

The membership-inference advantage is `attack accuracy - 0.5`: **attack accuracy minus one-half**, where one-half is random guessing on a balanced evaluation set.

In [ ]:
import os
import torch
import torch.optim as optim
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from safetensors.torch import save_file

from htb_ai_library import (
    CIFAR10CNN, compute_mia_advantage, evaluate_accuracy,
    get_cifar10_loaders, set_reproducibility, train_baseline_sgd,
    train_dp_sgd, use_htb_style,
)

In [ ]:
# Keep baseline and private runs comparable.
RANDOM_SEED = 1337
BATCH_SIZE = 256
BASELINE_EPOCHS = 20
DP_EPOCHS = 20
LEARNING_RATE = 0.1
MAX_GRAD_NORM = 1.0
DELTA = 1e-5

set_reproducibility(RANDOM_SEED)
use_htb_style()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for directory in ('figs', 'output', 'models'):
    os.makedirs(directory, exist_ok=True)
print(f'Using device: {DEVICE}')

## Load the controlled comparison data

CIFAR-10 has 50,000 training images and 10,000 test images. In this demonstration, training samples are known members and test samples are known non-members. The test set is never used to update the model.

In [ ]:
train_dataset, test_dataset, train_loader, test_loader = get_cifar10_loaders(
    batch_size=BATCH_SIZE, download=True
)
print(f'Training samples: {len(train_dataset):,}')
print(f'Test samples: {len(test_dataset):,}')

## Train and measure the non-private baseline

The baseline uses ordinary SGD with no clipping and no noise. It gives us a utility reference and a privacy-leakage reference. The overfitting gap is **training accuracy minus test accuracy**; a larger gap usually means more memorization.

In [ ]:
baseline_model = CIFAR10CNN().to(DEVICE)
baseline_model = train_baseline_sgd(
    baseline_model, train_loader, DEVICE,
    epochs=BASELINE_EPOCHS, learning_rate=LEARNING_RATE
)
train_acc_baseline = evaluate_accuracy(baseline_model, train_loader, DEVICE)
test_acc_baseline = evaluate_accuracy(baseline_model, test_loader, DEVICE)
mia_acc_baseline, mia_adv_baseline = compute_mia_advantage(
    baseline_model, train_loader, test_loader, DEVICE
)
print(f'Train accuracy: {train_acc_baseline:.2f}%')
print(f'Test accuracy: {test_acc_baseline:.2f}%')
print(f'Overfitting gap: {train_acc_baseline - test_acc_baseline:.2f} points')
print(f'MIA accuracy: {mia_acc_baseline:.4f}')
print(f'MIA advantage: {mia_adv_baseline:.4f}')
torch.save(baseline_model.state_dict(), 'output/baseline_model.pth')

## Train DP-SGD models at two privacy budgets

`make_private_with_epsilon()` asks Opacus to select the noise multiplier needed to reach a final `(epsilon, delta)` budget. It clips each per-example gradient to `MAX_GRAD_NORM`, adds Gaussian noise, and uses an RDP accountant to track privacy across all updates.

Each experiment gets a fresh loader and privacy engine because Opacus wraps data loading for privacy accounting.

In [ ]:
def train_private_model(target_epsilon):
    _, _, private_train_loader, private_test_loader = get_cifar10_loaders(
        batch_size=BATCH_SIZE, download=False
    )
    model = ModuleValidator.fix(CIFAR10CNN().to(DEVICE))
    optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)
    privacy_engine = PrivacyEngine(accountant='rdp')
    model, optimizer, private_train_loader = privacy_engine.make_private_with_epsilon(
        module=model, optimizer=optimizer, data_loader=private_train_loader,
        target_epsilon=target_epsilon, target_delta=DELTA,
        epochs=DP_EPOCHS, max_grad_norm=MAX_GRAD_NORM
    )
    model, final_epsilon = train_dp_sgd(
        model, private_train_loader, privacy_engine, optimizer, DEVICE
    )
    train_acc = evaluate_accuracy(model, private_train_loader, DEVICE)
    test_acc = evaluate_accuracy(model, private_test_loader, DEVICE)
    mia_acc, mia_adv = compute_mia_advantage(
        model, private_train_loader, private_test_loader, DEVICE
    )
    return model, private_train_loader, private_test_loader, {
        'epsilon': float(final_epsilon),
        'train_accuracy': float(train_acc),
        'test_accuracy': float(test_acc),
        'overfitting_gap': float(train_acc - test_acc),
        'mia_accuracy': float(mia_acc),
        'mia_advantage': float(mia_adv),
    }

dp_model_10, train_loader_10, test_loader_10, results_10 = train_private_model(10.0)
dp_model_3, train_loader_3, test_loader_3, results_3 = train_private_model(3.0)
print('epsilon=10:', results_10)
print('epsilon=3: ', results_3)

## Interpret the privacy-utility tradeoff

Expected direction, not a guaranteed exact output: the baseline should have the highest accuracy and largest overfitting gap; epsilon 10 should reduce leakage with a moderate accuracy cost; epsilon 3 should add more noise, reduce memorization further, and usually lower accuracy more.

This is the distinction between a formal DP guarantee and an empirical MIA result: epsilon/delta bound worst-case individual influence, while `compute_mia_advantage()` measures one specific attack on this dataset.

In [ ]:
# Opacus wraps the model; save the underlying model state.
torch.save(dp_model_10._module.state_dict(), 'output/dp_model_eps10.pth')
torch.save(dp_model_3._module.state_dict(), 'output/dp_model_eps3.pth')
# The HTB validator expects the epsilon=10 model in Safetensors format.
save_file(dp_model_10._module.state_dict(), 'models/dp_model.safetensors')

all_results = {
    'baseline': {
        'train_accuracy': float(train_acc_baseline),
        'test_accuracy': float(test_acc_baseline),
        'overfitting_gap': float(train_acc_baseline - test_acc_baseline),
        'mia_accuracy': float(mia_acc_baseline),
        'mia_advantage': float(mia_adv_baseline),
    },
    'dp_epsilon_10': results_10,
    'dp_epsilon_3': results_3,
}
print(all_results)

## Section 11: analyze the privacy-utility tradeoff

This section turns the three result dictionaries into a comparison table, figures, and a JSON artifact. The baseline is displayed with epsilon `infinity` because it has no formal privacy bound. JSON values are converted to ordinary Python floats so they serialize reliably.

In [ ]:
import json
from htb_ai_library import plot_accuracy_comparison, plot_privacy_utility_tradeoff

def print_comparison_table():
    rows = [
        ('Baseline', test_acc_baseline, mia_adv_baseline, 'infinity'),
        ('DP (epsilon=10)', results_10['test_accuracy'], results_10['mia_advantage'], results_10['epsilon']),
        ('DP (epsilon=3)', results_3['test_accuracy'], results_3['mia_advantage'], results_3['epsilon']),
    ]
    print(f'{"Model":<20} {"Test accuracy":<16} {"MIA advantage":<16} Epsilon')
    print('-' * 68)
    for name, accuracy, advantage, epsilon in rows:
        epsilon_text = epsilon if isinstance(epsilon, str) else f'{epsilon:.2f}'
        print(f'{name:<20} {accuracy:>8.2f}%       {advantage:>8.4f}       {epsilon_text}')

print_comparison_table()

In [ ]:
# These plots make the same relationship easier to see than a table alone.
plot_accuracy_comparison(
    test_acc_baseline, results_10['test_accuracy'], results_3['test_accuracy'],
    save_path='figs/dp_sgd_accuracy_comparison.png'
)
plot_privacy_utility_tradeoff(
    test_acc_baseline, results_10['test_accuracy'], results_3['test_accuracy'],
    mia_adv_baseline, results_10['mia_advantage'], results_3['mia_advantage'],
    results_10['epsilon'], results_3['epsilon'],
    save_path='figs/dp_sgd_privacy_utility_tradeoff.png'
)
print('Saved comparison figures under figs/.')

In [ ]:
# Persist the scalar findings for later comparison.
with open('output/results.json', 'w', encoding='utf-8') as handle:
    json.dump(all_results, handle, indent=2)
print('Saved output/results.json')

### Interpretation notes

The expected pattern is: lower epsilon reduces membership-inference advantage and the overfitting gap, but also lowers test accuracy. DP-SGD protects the influence of individual records; it does not prevent aggregate-statistics inference, model stealing, or adversarial examples. PATE, local DP, and federated learning use different privacy mechanisms and should be selected according to the threat model.

- Explain why a formal DP guarantee is a worst-case bound while MIA is an empirical attack result.
- Compare the marginal accuracy cost with the marginal privacy benefit as epsilon decreases.
- Keep submission controls disabled unless an HTB challenge explicitly requires them and the user authorizes submission.